In [1]:
import torch
import math
import numpy as np
import torch.nn as nn
import polars as pl
import xxhash
from tqdm import tqdm
from functools import reduce
from IPython.display import clear_output
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from torch.utils.tensorboard import SummaryWriter

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2026-05-07 22:15:06.929234: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 22:15:09.142865: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following ins

polars.config.Config

In [2]:
class QuerySoftMax(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x, y, group_ids):
        device = x.device
        _, group_id = torch.unique(group_ids, return_inverse=True)
        group_num = group_id.max().item() + 1
        obj_scores = torch.exp(x)
        group_scores = torch.zeros(group_num, device=device).index_add_(0, group_id, obj_scores)
        logits = obj_scores / group_scores[group_id]
        loss = -((torch.log(logits) * y).sum() / (y.sum()))
        return loss

In [3]:
class CrossNetworkLayer(nn.Module):
    def __init__(self, in_shape, rank):
        super().__init__()
        self.W1 = nn.Parameter(nn.init.kaiming_uniform_(torch.empty(in_shape, rank), a=math.sqrt(5)))
        self.W2 = nn.Parameter(nn.init.kaiming_uniform_(torch.empty(rank, in_shape), a=math.sqrt(5)))
        bound = 1 / math.sqrt(rank)
        self.bias = nn.Parameter(torch.empty(in_shape).uniform_(-bound, bound))
        
    def forward(self, x0, x_prev):
        out = x0 * ((x_prev @ self.W1) @ self.W2 + self.bias) + x_prev
        return out

In [4]:
class DeepNetworkLayer(nn.Module):
    def __init__(self, in_shape, out_shape, rank):
        super().__init__()
        self.W1 = nn.Parameter(nn.init.kaiming_uniform_(torch.empty(in_shape, rank), a=math.sqrt(5)))
        self.W2 = nn.Parameter(nn.init.kaiming_uniform_(torch.empty(rank, out_shape), a=math.sqrt(5)))
        bound = 1 / math.sqrt(rank)
        self.bias = nn.Parameter(torch.empty(out_shape).uniform_(-bound, bound))
        self.relu = nn.ReLU()
        
    def forward(self, x0):
        out = self.relu((x0 @ self.W1) @ self.W2 + self.bias)
        return out

In [5]:
dataset = pl.read_parquet("/home/jupyter/filestore/storage/datasets/train_dataset_with_feats.parquet")

In [9]:
user_segments = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
)
    
user_segments.select("user_id").unique().shape

(57122, 1)

In [6]:
top_features = [
    "event_id=item_view_user_id_cnt", "event_id=item_view_user_id_distinct",
    "event_id=item_view_user_id_distinct_wnd=7", "event_id=item_view_user_id_cnt_wnd=7",
    "event_id=item_like_user_id_cnt", "event_id=item_like_user_id_distinct",
    "event_id=item_like_user_id_distinct_wnd=7", "event_id=item_view_item_id_cnt_by=c2_name",
    "event_id=item_view_item_id_cnt_wnd=7_by=brand_name", "event_id=item_view_item_id_cnt_by=brand_name",
    "event_id=buy_comp_item_id_cnt_wnd=7_by=brand_name"
]

In [7]:
class TrainDataset(Dataset):
    def __init__(self, dataset, num_feature_names, text_feature_names):
        self.groups = dataset.partition_by("user_id", as_dict=False)
        self.num_feature_names = num_feature_names
        self.text_feature_names = text_feature_names

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        group_df = self.groups[idx]

        num_features = torch.tensor(
            group_df.select(*self.num_feature_names).to_numpy(),
            dtype=torch.float32
        )
        
        item_ids = torch.tensor(
            group_df.select("item_id").to_numpy(),
            dtype=torch.int64
        )
        
        text_features = group_df.select(*self.text_feature_names).to_numpy()

        target = torch.tensor(
            group_df["target"].to_numpy(),
            dtype=torch.int8
        )

        return {
            "num_features": num_features,
            "text_features": text_features, 
            "target": target,
            "user_id": group_df["user_id"][0],
            "item_id": item_ids
        }

In [8]:
def collate_fn(batch):
    user_ids = [[sample["user_id"]] * sample["target"].shape[0] for sample in batch]
    user_ids = torch.tensor(reduce(lambda x, y: x + y, user_ids))
    num_features = torch.cat([sample["num_features"] for sample in batch], dim=0)
    text_features = np.concatenate([sample["text_features"] for sample in batch], axis=0)
    target = torch.cat([sample["target"] for sample in batch], dim=0)
    item_ids = torch.cat([sample["item_id"] for sample in batch], dim=0).flatten()
    return [user_ids, num_features, text_features, target, item_ids]

In [9]:
num_feature_names = top_features

text_feature_names = [
    "c2_name", "brand_name", "item_condition_name"
]

train_data = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
    .filter(pl.col("user_segment").is_in(list(range(3, 10))))
    .select("user_id", "target", "item_id", *num_feature_names, *text_feature_names)
    .with_columns(
        *[pl.col(feat_name).fill_null("NO_INFO") for feat_name in text_feature_names],
        *[pl.col(feat_name).fill_null(0) for feat_name in num_feature_names]
    )
)

val_data = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
    .filter(pl.col("user_segment").is_in(list(range(2, 3))))
    .select("user_id", "target", "item_id", *num_feature_names, *text_feature_names)
    .with_columns(
        *[pl.col(feat_name).fill_null("NO_INFO") for feat_name in text_feature_names],
        *[pl.col(feat_name).fill_null(0) for feat_name in num_feature_names]
    )
)

test_data = (
    dataset
    .with_columns(
        pl.col("user_id").apply(lambda x: abs(xxhash.xxh64(str(x), seed=42).intdigest()) % 100).alias("user_segment")
    )
    .filter(pl.col("user_segment").is_in(list(range(2))))
    .select("user_id", "target", "item_id", *num_feature_names, *text_feature_names)
    .with_columns(
        *[pl.col(feat_name).fill_null("NO_INFO") for feat_name in text_feature_names],
        *[pl.col(feat_name).fill_null(0) for feat_name in num_feature_names]
    )
)

print("train shape and users cnt: ", train_data.shape[0], train_data.select("user_id").unique().shape[0])
print("val shape and users cnt: ", val_data.shape[0], val_data.select("user_id").unique().shape[0])
print("test shape and users cnt: ", test_data.shape[0], test_data.select("user_id").unique().shape[0])

train shape and users cnt:  1007600 40052
val shape and users cnt:  139737 5649
test shape and users cnt:  278321 11421


In [10]:
text_encoder = SentenceTransformer("all-MiniLM-L6-v2")

for p in text_encoder.parameters():
    p.requires_grad = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 317.90it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
train_dataset = TrainDataset(train_data, num_feature_names, text_feature_names)
train_loader = DataLoader(train_dataset, batch_size=128, collate_fn=collate_fn)

val_dataset = TrainDataset(val_data, num_feature_names, text_feature_names)
val_loader = DataLoader(val_dataset, batch_size=128, collate_fn=collate_fn)

test_dataset = TrainDataset(test_data, num_feature_names, text_feature_names)
test_loader = DataLoader(test_dataset, batch_size=128, collate_fn=collate_fn)

text_emb_dim = text_encoder.get_sentence_embedding_dimension()
nn_input_shape = text_emb_dim * len(text_feature_names) + len(num_feature_names)
print("Ranker input shape: ", nn_input_shape)

Ranker input shape:  1163


In [12]:
class NeuralRanker(nn.Module):
    def __init__(self, in_shape, cross_layers_num, deep_layers_num, rank, text_encoder, mean, std):
        super().__init__()
        self.cross_network = nn.ModuleList([CrossNetworkLayer(in_shape, rank) for _ in range(cross_layers_num)])
        self.deep_network = nn.Sequential(*[DeepNetworkLayer(in_shape, in_shape, rank) for _ in range(deep_layers_num)])
        self.lin1 = nn.Linear(2 * in_shape, in_shape)
        self.relu = nn.ReLU()
        self.lin2 = nn.Linear(in_shape, 1)
        self.text_encoder = text_encoder
        self.register_buffer("mean", torch.tensor(mean))
        self.register_buffer("std", torch.tensor(std))
        
    def forward(self, x):
        user_ids, num_features, text_features, target, item_ids = x
        device = user_ids.device
        self.mean.to(device)
        self.std.to(device)
        normalized_num_feats = ((num_features - self.mean) / self.std).to(torch.float32)
        text_embs = []
        for text_feat_idx in range(text_features.shape[1]):
            text_emb = self.text_encoder.encode(
                text_features[:, text_feat_idx], batch_size=1024,
                normalize_embeddings=True, convert_to_tensor=True
            )
            text_embs.append(text_emb)
        final_text_emb = torch.cat(text_embs, dim=-1)
        x0 = torch.cat([normalized_num_feats, final_text_emb], dim=-1)
        x_prev = x0
        deep_out = self.deep_network(x0)
        for cross_layer in self.cross_network:
            x_prev = cross_layer(x0, x_prev)
        x1 = torch.cat([deep_out, x_prev], dim=-1) 
        out = self.lin2(self.relu(self.lin1(x1))).flatten()
        return out

In [13]:
mean = train_data.select(*[pl.mean(feat_name) for feat_name in num_feature_names]).to_numpy()
std = train_data.select(*[pl.std(feat_name) for feat_name in num_feature_names]).to_numpy()

In [14]:
model = NeuralRanker(nn_input_shape, 1, 1, 32, text_encoder, mean, std)
print("model parameters: ", sum([p.numel() for p in model.parameters() if p.requires_grad]))

model parameters:  2858655


In [18]:
model.load_state_dict(torch.load("ranker_v1.pth", map_location=torch.device("cpu")))

/tmp/ipykernel_2933/3948417574.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("ranker_v1.pth", map_location=torch.device("cpu")))


<All keys matched successfully>

In [47]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
criterion = QuerySoftMax()
writer = SummaryWriter()

print(f"device: {device}")

device: cuda:0


In [26]:
def calc_ranking_metrics(user_ids, item_ids, target, scores, k_values):
    metrics = defaultdict(list)
    
    scores_df = pl.DataFrame({
        "user_id": user_ids,
        "item_id": item_ids,
        "score": scores,
        "target": target,
    })
    
    scores_with_rn = (
        scores_df
        .sort(["user_id", "score", "item_id"], descending=[False, True, False])
        .with_row_count("global_rank")
        .with_columns(
            (
                pl.col("global_rank")
                - pl.col("global_rank").first().over("user_id")
                + 1
            ).alias("rn")
        )
    )
    
    for k in k_values:
        topk_scores = (
            scores_with_rn
            .with_columns(
                  (pl.col("rn") <= k).cast(pl.Int8).alias("topk")
            )
        )
        
        metrics_at_k = (
            topk_scores
            .groupby("user_id")
            .agg(
                pl.col("target").sum().alias("relevant_cnt"),
                pl.col("target").filter(pl.col("topk") == 1).sum().alias("topk_relevant"),
            )
            .with_columns(
                (pl.col("topk_relevant") > 0).cast(pl.Float32).alias("hitrate"),
                (pl.col("topk_relevant") / k).alias("precision"),
                (pl.col("topk_relevant") / pl.col("relevant_cnt")).alias("recall")
            )
            .select(
                pl.mean("hitrate").alias("hitrate"),
                pl.mean("precision").alias("precision"),
                pl.mean("recall").alias("recall")
            )
        )
        
        metrics["hitrate"].append(metrics_at_k["hitrate"][0])
        metrics["precision"].append(metrics_at_k["precision"][0])
        metrics["recall"].append(metrics_at_k["recall"][0])
        
    return metrics

In [49]:
def train_epoch(model, dataloader, writer, criterion, optimizer, epoch, device="cpu", k_values=None):
    model.train()
    running_loss = 0
    user_ids = []
    item_ids = []
    target = []
    scores = []
    for batch in tqdm(dataloader, total=len(dataloader), desc=f'Epoch: {epoch}, Training Batch', dynamic_ncols=True):
        batch[0] = batch[0].to(device)
        batch[1] = batch[1].to(device)
        batch[3] = batch[3].to(device)
        batch_scores = model(batch)
        loss = criterion(batch_scores, batch[3], batch[0])
        if epoch > 0:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        running_loss += loss.detach().cpu().item()
        #print(loss)
        user_ids.append(batch[0].detach().cpu().numpy())
        item_ids.append(batch[4].detach().cpu().numpy())
        scores.append(batch_scores.detach().cpu().numpy())
        target.append(batch[3].detach().cpu().numpy())
    user_ids = np.concatenate(user_ids, axis=0)
    item_ids = np.concatenate(item_ids, axis=0)
    scores = np.concatenate(scores, axis=0)
    target = np.concatenate(target, axis=0)
    if k_values is not None:
        ranking_metrics = calc_ranking_metrics(user_ids, item_ids, target, scores, k_values)
        for metric in ranking_metrics:
            for i, k in enumerate(k_values):
                writer.add_scalar(f'{metric}@{k}/Train', ranking_metrics[metric][i], epoch + 1)
    writer.add_scalar('Loss/Train', running_loss / len(dataloader), epoch + 1)
    return running_loss / len(dataloader), optimizer

In [50]:
def eval_epoch(model, dataloader, writer, criterion, optimizer, epoch, device="cpu", k_values=None):
    model.eval()
    running_loss = 0
    user_ids = []
    item_ids = []
    target = []
    scores = []
    with torch.no_grad():
        for batch in tqdm(dataloader, total=len(dataloader), desc=f'Epoch: {epoch}, Validation Batch', dynamic_ncols=True):
            batch[0] = batch[0].to(device)
            batch[1] = batch[1].to(device)
            batch[3] = batch[3].to(device)
            batch_scores = model(batch)
            loss = criterion(batch_scores, batch[3], batch[0])
            running_loss += loss.detach().cpu().item()
            user_ids.append(batch[0].detach().cpu().numpy())
            item_ids.append(batch[4].detach().cpu().numpy())
            scores.append(batch_scores.detach().cpu().numpy())
            target.append(batch[3].detach().cpu().numpy())
        user_ids = np.concatenate(user_ids, axis=0)
        item_ids = np.concatenate(item_ids, axis=0)
        scores = np.concatenate(scores, axis=0)
        target = np.concatenate(target, axis=0)
        if k_values is not None:
            ranking_metrics = calc_ranking_metrics(user_ids, item_ids, target, scores, k_values)
            for metric in ranking_metrics:
                for i, k in enumerate(k_values):
                    writer.add_scalar(f'{metric}@{k}/Val', ranking_metrics[metric][i], epoch + 1)
        writer.add_scalar('Loss/Val', running_loss / len(dataloader), epoch + 1)
    return running_loss / len(dataloader), optimizer

In [51]:
def train(model, writer, train_dataloader, val_dataloader, criterion, optimizer, epochs=20, device="cpu", k_values=None):
    model = model.to(device)
    for i in range(epochs):
        clear_output(wait=True)
        train_loss, optimizer = train_epoch(
            model, train_dataloader, writer, criterion, optimizer, i,
            device=device, k_values=k_values
        )
        print(f"Epoch: {i + 1}, Train Loss: {round(train_loss, 4)}")
        eval_loss, optimizer = eval_epoch(
            model, val_dataloader, writer, criterion, optimizer, i,
            device=device, k_values=k_values
        )
        print(f"Epoch: {i + 1}, Val Loss: {round(eval_loss, 4)}")
    return model

In [52]:
train(model, writer, train_loader, val_loader, criterion, optimizer, device=device, k_values=[1, 10])

Epoch: 19, Training Batch: 100%|██████████| 313/313 [02:35<00:00,  2.02it/s]


Epoch: 20, Train Loss: 3.1415


Epoch: 19, Validation Batch: 100%|██████████| 45/45 [00:21<00:00,  2.08it/s]

Epoch: 20, Val Loss: 3.0921


NeuralRanker(
  (cross_network): ModuleList(
    (0): CrossNetworkLayer()
  )
  (deep_network): Sequential(
    (0): DeepNetworkLayer(
      (relu): ReLU()
    )
  )
  (lin1): Linear(in_features=2326, out_features=1163, bias=True)
  (relu): ReLU()
  (lin2): Linear(in_features=1163, out_features=1, bias=True)
  (text_encoder): SentenceTransformer(
    (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
    (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
    (2): Normalize()
  )
)

In [53]:
torch.save(model.state_dict(), "ranker_v1.pth")

In [24]:
def inference(model, dataloader, k_values, device="cpu"):
    model.eval()
    metrics = {}
    user_ids = []
    item_ids = []
    target = []
    scores = []
    with torch.no_grad():
        for batch in tqdm(dataloader, total=len(dataloader), desc='Inference', dynamic_ncols=True):
            batch[0] = batch[0].to(device)
            batch[1] = batch[1].to(device)
            batch[3] = batch[3].to(device)
            batch_scores = model(batch)
            user_ids.append(batch[0].detach().cpu().numpy())
            item_ids.append(batch[4].detach().cpu().numpy())
            scores.append(batch_scores.detach().cpu().numpy())
            target.append(batch[3].detach().cpu().numpy())
        user_ids = np.concatenate(user_ids, axis=0)
        item_ids = np.concatenate(item_ids, axis=0)
        scores = np.concatenate(scores, axis=0)
        target = np.concatenate(target, axis=0)
        if k_values is not None:
            ranking_metrics = calc_ranking_metrics(user_ids, item_ids, target, scores, k_values)
            for metric in ranking_metrics:
                for i, k in enumerate(k_values):
                    metrics[f'{metric}@{k}'] = ranking_metrics[metric][i]
    return metrics

In [27]:
test_metrics = inference(model, test_loader, [1, 10])

Inference: 100%|██████████| 90/90 [14:39<00:00,  9.77s/it]


In [28]:
test_metrics

{'hitrate@1': 0.5494264960289001,
 'hitrate@10': 0.9978985786437988,
 'precision@1': 0.5494264950529726,
 'precision@10': 0.22506785745556426,
 'recall@1': 0.2971349405622359,
 'recall@10': 0.8625140365229117}